### Import packages

In [3]:
import pandas as pd
from scipy.stats import spearmanr
from scipy.stats import chi2_contingency
import numpy as np


### EDA

1. LOAD DATA

In [4]:
# load data
data = pd.read_csv('data_2024_reduced_columns.csv')
data['B2'].unique()

array(['Doing okay', 'Living comfortably', 'Just getting by',
       'Finding it difficult to get by'], dtype=object)

2. BINARY VARIABLE TRANSFORMATON

In [25]:
# map columms from categorial to ordianal
B2_mapping = {
    "Finding it difficult to get by": 1,
    "Just getting by": 2,
    "Doing okay": 3,
    "Living comfortably": 4
}

BNPL1_mapping = {
  "No": 0,
  "Yes": 1
}


In [26]:
# convert columns to ordinal
data["B2_ord"] = data["B2"].map(B2_mapping)
data["BNPL1_ord"] = data["BNPL1"].map(BNPL1_mapping)

In [15]:
# spearman relationship
spearmanr(data["B2_ord"], data["BNPL1"], nan_policy="omit")

SignificanceResult(statistic=-0.18292396676570352, pvalue=5.549600237840652e-93)

3. ONE-HOT ENCODE TRANSFORMATION

In [ ]:
B2_dummies = pd.get_dummies(data["B2"], prefix="B2")

# Clean column names
B2_dummies.columns = (
    B2_dummies.columns
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("to_get_by", "to_get_by")  # optional tweak
)

data = pd.concat([data, B2_dummies], axis=1)

,B2,B2_Doing okay
0,Doing okay,True
1,Living comfortably,False
6,Just getting by,False
12,Finding it difficult to get by,False


4. CREATE NEW COLUMNS THAT COMBINES MULTIPLE COLUMNS

In [42]:
# combine x12_abcdefg to column financial_concern_flag
cols = ["X12_a", "X12_b", "X12_c", "X12_d", "X12_e", "X12_f", "X12_g"]

data["financial_concern_flag"] = np.where(
    data[cols].isna().all(axis=1),  # all missing → keep NaN
    np.nan,
    data[cols].isin(["Major concern", "Minor concern"]).any(axis=1).astype(int)
)

In [ ]:
# create new column financial_concern_count to add number of conerns in X12 a to g
data["financial_concern_count"] = np.where(
    data[cols].isna().all(axis=1),
    np.nan,
    data[cols].isin(["Minor concern", "Major concern"]).sum(axis=1)
)

In [53]:
# create new column financial_concern_score to add up number of conerns in X12 a to g with major concer=2, minor concern = 1
mapping = {
    "Not a concern": 0,
    "Minor concern": 1,
    "Major concern": 2
}

data["financial_concern_score"] = np.where(
    data[cols].isna().all(axis=1),
    np.nan,
    data[cols].replace(mapping).sum(axis=1)
)

/var/folders/cr/df6gvtx90rz5bg2m0clb1gym0000gn/T/ipykernel_69592/1170674784.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[cols].replace(mapping).sum(axis=1)


In [ ]:
# create financial concern flag for each question X12 abcdefg
mapping = {
    "Not a concern": 0,
    "Minor concern": 1,
    "Major concern": 1
}

cols = ["X12_a", "X12_b", "X12_c", "X12_d", "X12_e", "X12_f", "X12_g"]

for col in cols:
    data[f"financial_concern_flag_{col}"] = np.where(
    data[col].isna(),
    np.nan,
    data[col].replace(mapping)
    )


/var/folders/cr/df6gvtx90rz5bg2m0clb1gym0000gn/T/ipykernel_69592/2038383620.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[col].replace(mapping)
/var/folders/cr/df6gvtx90rz5bg2m0clb1gym0000gn/T/ipykernel_69592/2038383620.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[col].replace(mapping)
/var/folders/cr/df6gvtx90rz5bg2m0clb1gym0000gn/T/ipykernel_69592/2038383620.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitl

In [15]:
# create financial concern score for each question X12 abcdefg
mapping = {
    "Not a concern": 0,
    "Minor concern": 1,
    "Major concern": 2
}

cols = ["X12_a", "X12_b", "X12_c", "X12_d", "X12_e", "X12_f", "X12_g"]

for col in cols:
    data[f"financial_concern_{col}"] = data[col].replace(mapping)

/var/folders/cr/df6gvtx90rz5bg2m0clb1gym0000gn/T/ipykernel_69592/1267505192.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[f"financial_concern_{col}"] = data[col].replace(mapping)
/var/folders/cr/df6gvtx90rz5bg2m0clb1gym0000gn/T/ipykernel_69592/1267505192.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[f"financial_concern_{col}"] = data[col].replace(mapping)
/var/folders/cr/df6gvtx90rz5bg2m0clb1gym0000gn/T/ipykernel_69592/1267505192.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will

In [66]:
# create renting flag

cols = ['R1_a','R1_b','R1_c','R1_d','R1_e','R1_f','R1_g']

for col in cols:
    data["renting_flag"] = np.where(
        data[cols].isna().all(axis=1),  # all missing → 0
        0,
        (data[cols]=='Yes').any(axis=1).astype(int)
    )

5. CHI-SQUARE TEST

In [60]:
# define cramers_v function
def compute_cramers_v(chi2, table):
    n = table.values.sum()
    r, c = table.shape
    k = min(r, c)
    return np.sqrt(chi2 / (n * (k - 1)))

# variables to test against BNPL1
predictors = ["B2",
              "financial_concern_flag", 'financial_concern_count', 'financial_concern_score',
              'financial_concern_X12_a', 'financial_concern_X12_b', 'financial_concern_X12_c', 'financial_concern_X12_d', 'financial_concern_X12_e', 'financial_concern_X12_f', 'financial_concern_X12_g',
              'financial_concern_flag_X12_a', 'financial_concern_flag_X12_b', 'financial_concern_flag_X12_c', 'financial_concern_flag_X12_d', 'financial_concern_flag_X12_e', 'financial_concern_flag_X12_f', 'financial_concern_flag_X12_g',
              'ppgender',
              'ppethm',
              'ppmarit5',
              'D22_i']

results = []

for var in predictors:
    # remove na
    subset = data[[var, "BNPL1"]].dropna()

    # contingency table
    table = pd.crosstab(subset[var], subset["BNPL1"])

    # chi-square test
    chi2, p, dof, expected = chi2_contingency(table)

    # cramér's v
    v = compute_cramers_v(chi2, table)

    results.append({
        "variable": var,
        "chi2": chi2,
        "p_value": p,
        "cramers_v": v
    })

# convert to DataFrame for display
results_df = pd.DataFrame(results)
print(results_df)


                        variable        chi2       p_value  cramers_v
0                             B2  424.538073  1.070648e-91   0.185821
1         financial_concern_flag   27.115867  1.916187e-07   0.066536
2        financial_concern_count  175.799385  1.501158e-34   0.169416
3        financial_concern_score  219.365463  5.930431e-39   0.189248
4        financial_concern_X12_a   56.043810  6.764587e-13   0.095656
5        financial_concern_X12_b   68.986489  1.046584e-15   0.106128
6        financial_concern_X12_c  107.524868  4.479934e-24   0.132496
7        financial_concern_X12_d   66.897785  2.973929e-15   0.104509
8        financial_concern_X12_e  228.753785  2.122007e-50   0.193255
9        financial_concern_X12_f   67.356105  2.364872e-15   0.104866
10       financial_concern_X12_g  121.224791  4.746475e-27   0.140683
11  financial_concern_flag_X12_a   51.428376  7.425742e-13   0.091632
12  financial_concern_flag_X12_b   13.321498  2.623805e-04   0.046636
13  financial_concer